# 05 — ResNet-ish (CNN) + LSTM Baseline (MCMIPF on-the-fly)

## Purpose
Train a spatio-temporal baseline model:
 - Spatial encoder: small ResNet-style CNN over PATCH×PATCH
 - Temporal encoder: LSTM over history window
 - Target: GHI at t_target (6h ahead)

## Imports and settings

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import time
import random
from functools import lru_cache

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path("..").resolve()
DATASET_ROOT = PROJECT_ROOT / "data" / "datasets" / "manifest_v1"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

## Load manifests + meta

In [ ]:
SITE = "uniandes"  # change when needed

SITE_DIR = DATASET_ROOT / SITE
assert SITE_DIR.exists(), f"Missing dataset directory: {SITE_DIR}"

with open(SITE_DIR / "dataset_meta.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

print("Loaded dataset_meta.json:")
print(json.dumps(meta, indent=2))

MCMIPF_ROOT = Path(meta["mcmipf_root"])
FREQ_MIN = int(meta["freq_min"])
H = int(meta["horizon_steps"])
L = int(meta["history_steps"])

GRID_SIZE = int(meta["grid_size"])
SITE_CENTER = (int(meta["site_center_pix"]["row"]), int(meta["site_center_pix"]["col"]))


### Spatial

In [ ]:
PATCH = 16  # keep aligned with your current run; you can switch to 32 later
HALF = PATCH // 2

train_man = pd.read_parquet(SITE_DIR / "manifest_train.parquet")
val_man   = pd.read_parquet(SITE_DIR / "manifest_val.parquet")
test_man  = pd.read_parquet(SITE_DIR / "manifest_test.parquet")

print("train:", train_man.shape)
print("val:  ", val_man.shape)
print("test: ", test_man.shape)

## Reproducibility

In [ ]:
SEED = int(meta.get("seed", 42))

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Determinism (slower, but reproducible)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("SEED:", SEED)


## Debug option

In [ ]:
DEBUG = False
DEBUG_TRAIN_N = 4000
DEBUG_VAL_N   = 1200
DEBUG_TEST_N  = 1200

if DEBUG:
    train_man = train_man.sample(n=min(DEBUG_TRAIN_N, len(train_man)), random_state=SEED).reset_index(drop=True)
    val_man   = val_man.sample(n=min(DEBUG_VAL_N, len(val_man)), random_state=SEED).reset_index(drop=True)
    test_man  = test_man.sample(n=min(DEBUG_TEST_N, len(test_man)), random_state=SEED).reset_index(drop=True)

print("DEBUG:", DEBUG)
print("train:", train_man.shape)
print("val:  ", val_man.shape)
print("test: ", test_man.shape)

## Baseline: Persistence on the same manifests
$$\hat{y}(t+H) = ghi(t)$$

In [ ]:
GROUND_DIR = PROJECT_ROOT / "data" / "ground_aligned"
ground_path = GROUND_DIR / f"ground_10min_utc_{SITE}.parquet"
assert ground_path.exists(), f"Missing ground parquet for {SITE}: {ground_path}"

ground = pd.read_parquet(ground_path)
assert "ghi" in ground.columns, "Ground dataset missing 'ghi'"
assert str(ground.index.tz) == "UTC", "Ground index must be UTC"

def eval_persistence(manifest: pd.DataFrame, ground_df: pd.DataFrame) -> dict:
    t_label = pd.to_datetime(manifest["t_label"], utc=True)
    y_true = manifest["y"].astype(float).to_numpy()
    y_hat = ground_df.reindex(t_label)["ghi"].to_numpy()  # persistence: y(t)

    mask = np.isfinite(y_true) & np.isfinite(y_hat)
    y_true = y_true[mask]
    y_hat = y_hat[mask]

    rmse = float(np.sqrt(np.mean((y_true - y_hat) ** 2)))
    mae  = float(np.mean(np.abs(y_true - y_hat)))
    return {"n": int(mask.sum()), "rmse": rmse, "mae": mae}

print("=== Persistence baseline (same manifests) ===")
print("train:", eval_persistence(train_man, ground))
print("val:  ", eval_persistence(val_man, ground))
print("test: ", eval_persistence(test_man, ground))

### Target normalization

In [ ]:
y_train = train_man["y"].astype(float).to_numpy()
Y_MEAN = float(np.mean(y_train))
Y_STD  = float(np.std(y_train) + 1e-6)

def norm_y(y: float) -> float:
    return (y - Y_MEAN) / Y_STD

def denorm_y(arr: np.ndarray) -> np.ndarray:
    return arr * Y_STD + Y_MEAN

print("Target stats from train:")
print("  mean:", Y_MEAN)
print("  std: ", Y_STD)
print("  percentiles:", np.percentile(y_train, [0, 50, 90, 95, 99]).tolist())


## MCMIPF

In [ ]:
def hour_path_for_timestamp(t: pd.Timestamp) -> Path:
    ymd = t.strftime("%Y%m%d")
    hh = t.strftime("%H")
    year = t.strftime("%Y")
    month = t.strftime("%m")
    fname = f"{ymd}_{hh}_MCMIPF.npz"
    path = MCMIPF_ROOT / year / month / fname
    return path

def slot_for_timestamp(t: pd.Timestamp) -> int:
    return int(t.strftime("%M")) // 10  # 0..5

def extract_patch(frame_c_hw: np.ndarray, center_rc: tuple[int, int], patch: int) -> np.ndarray:
    r0, c0 = center_rc
    half = patch // 2
    r1, r2 = r0 - half, r0 + half
    c1, c2 = c0 - half, c0 + half

    if (r1 < 0) or (c1 < 0) or (r2 > GRID_SIZE) or (c2 > GRID_SIZE):
        raise ValueError(f"Patch exceeds bounds: rows [{r1},{r2}) cols [{c1},{c2})")

    return frame_c_hw[:, r1:r2, c1:c2]  # (C, P, P)

In [ ]:
@lru_cache(maxsize=256)  # tune based on RAM; 256 hour-files ~ reasonable starting point
def load_hour_npz(path_str: str) -> np.ndarray:
    path = Path(path_str)
    with np.load(path) as data:
        arr = data["mcmipf"].astype(np.float32)  # (6,16,256,256)
    return arr

### Dataset

In [ ]:
class PatchSeqDataset(Dataset):
    def __init__(self, manifest: pd.DataFrame, site_center: tuple[int, int]):
        self.man = manifest.reset_index(drop=True)
        self.site_center = site_center

    def __len__(self) -> int:
        return len(self.man)

    def __getitem__(self, i: int):
        row = self.man.iloc[i]
        y = norm_y(float(row["y"]))
        history_ts = row["history_ts"]

        # parquet sometimes stores lists as strings
        if isinstance(history_ts, str):
            history_ts = json.loads(history_ts)

        xs = []
        for ts_str in history_ts:
            t = pd.to_datetime(ts_str, utc=True)
            p = hour_path_for_timestamp(t)
            slot = slot_for_timestamp(t)

            tensor = load_hour_npz(str(p))    # (6,16,256,256)
            frame = tensor[slot]              # (16,256,256)

            patch = extract_patch(frame, self.site_center, PATCH)  # (16,P,P)
            patch = np.nan_to_num(patch, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
            xs.append(patch)

        x_seq = np.stack(xs, axis=0)  # (L, C, P, P)
        return torch.from_numpy(x_seq), torch.tensor(y, dtype=torch.float32)

train_ds = PatchSeqDataset(train_man, SITE_CENTER)
val_ds   = PatchSeqDataset(val_man, SITE_CENTER)
test_ds  = PatchSeqDataset(test_man, SITE_CENTER)

print("train_ds:", len(train_ds), "val_ds:", len(val_ds), "test_ds:", len(test_ds))

## Dataloaders

In [ ]:
BATCH_SIZE = 8
NUM_WORKERS = 4

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"), persistent_workers=(NUM_WORKERS > 0)
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"), persistent_workers=(NUM_WORKERS > 0)
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"), persistent_workers=(NUM_WORKERS > 0)
)

## Metrics

In [ ]:
def metrics_from_arrays(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = y_true.astype(np.float64)
    y_pred = y_pred.astype(np.float64)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    return {"n": int(mask.sum()), "rmse": rmse, "mae": mae}

@torch.no_grad()
def eval_model(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    ys, yhats = [], []
    for x_seq, y in loader:
        x_seq = x_seq.to(DEVICE, non_blocking=True)  # (B,L,C,P,P)
        y = y.to(DEVICE, non_blocking=True)
        yhat = model(x_seq)
        ys.append(y.detach().cpu().numpy())
        yhats.append(yhat.detach().cpu().numpy())

    y = np.concatenate(ys)
    yhat = np.concatenate(yhats)

    y = denorm_y(y)
    yhat = denorm_y(yhat)

    return metrics_from_arrays(y, yhat)

## Model

Small ResNet-ish Encoder + LSTM + Head


In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)

        self.down = None
        if stride != 1 or in_ch != out_ch:
            self.down = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.down is not None:
            identity = self.down(identity)
        out = self.relu(out + identity)
        return out

class SmallResNetEncoder(nn.Module):
    """
    Small-input ResNet-ish encoder:
    - stem: 3x3 stride1 (no maxpool)
    - a few residual stages with light downsampling
    - global average pool -> embedding
    """
    def __init__(self, in_ch: int = 16, base: int = 32, emb_dim: int = 128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, base, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base),
            nn.ReLU(inplace=True),
        )

        # For PATCH=16, downsample gently: 16 -> 8 -> 4
        self.layer1 = nn.Sequential(
            BasicBlock(base, base, stride=1),
            BasicBlock(base, base, stride=1),
        )
        self.layer2 = nn.Sequential(
            BasicBlock(base, base * 2, stride=2),
            BasicBlock(base * 2, base * 2, stride=1),
        )
        self.layer3 = nn.Sequential(
            BasicBlock(base * 2, base * 4, stride=2),
            BasicBlock(base * 4, base * 4, stride=1),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Linear(base * 4, emb_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, P, P)
        h = self.stem(x)
        h = self.layer1(h)
        h = self.layer2(h)
        h = self.layer3(h)
        h = self.pool(h).squeeze(-1).squeeze(-1)  # (B, base*4)
        z = self.proj(h)                          # (B, emb_dim)
        return z

class ResNetLSTM(nn.Module):
    def __init__(self, in_ch: int = 16, patch: int = 16, emb_dim: int = 128, hidden_t: int = 128):
        super().__init__()
        self.encoder = SmallResNetEncoder(in_ch=in_ch, base=32, emb_dim=emb_dim)
        self.lstm = nn.LSTM(input_size=emb_dim, hidden_size=hidden_t, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_t, hidden_t),
            nn.ReLU(),
            nn.Linear(hidden_t, 1),
        )

    def forward(self, x_seq: torch.Tensor) -> torch.Tensor:
        # x_seq: (B, L, C, P, P)
        B, Ls, C, P, P2 = x_seq.shape
        assert P == P2, "Patch must be square"

        # Fully batched spatial encoding:
        x = x_seq.reshape(B * Ls, C, P, P)
        z = self.encoder(x)                # (B*L, emb_dim)
        z = z.reshape(B, Ls, -1)           # (B, L, emb_dim)

        out, _ = self.lstm(z)              # (B, L, hidden_t)
        last = out[:, -1]                  # (B, hidden_t)
        yhat = self.head(last).squeeze(-1) # (B,)
        return yhat

# %%
model = ResNetLSTM(in_ch=16, patch=PATCH, emb_dim=128, hidden_t=128).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

n_params = sum(p.numel() for p in model.parameters())
print("Model params:", n_params / 1e6, "M")

## Sanity check

In [ ]:
row = train_man.iloc[0]
history_ts = row["history_ts"]
if isinstance(history_ts, str):
    history_ts = json.loads(history_ts)

t0 = time.time()
for ts_str in history_ts[:10]:
    t = pd.to_datetime(ts_str, utc=True)
    p = hour_path_for_timestamp(t)
    tensor = load_hour_npz(str(p))
    _ = tensor[slot_for_timestamp(t)]
print("10 frames load time (s):", time.time() - t0)

## Training

In [ ]:
EPOCHS = 30
PATIENCE = 5
MIN_DELTA = 2.0
BEST_PATH = SITE_DIR / "best_model_resnet_lstm.pt"

train_log = []
best_rmse = float("inf")
best_epoch = 0
bad_epochs = 0

t_train0 = time.time()

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tr_losses = []

    for x_seq, y in train_loader:
        x_seq = x_seq.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        yhat = model(x_seq)
        loss = loss_fn(yhat, y)
        loss.backward()
        opt.step()

        tr_losses.append(loss.item())

    val_metrics = eval_model(model, val_loader)
    val_rmse = float(val_metrics["rmse"])
    val_mae  = float(val_metrics["mae"])

    epoch_out = {
        "epoch": epoch,
        "train_mse": float(np.mean(tr_losses)),
        "val_rmse": val_rmse,
        "val_mae": val_mae,
        "epoch_seconds": float(time.time() - t0),
    }
    train_log.append(epoch_out)

    improved = (best_rmse - val_rmse) > MIN_DELTA

    if improved:
        best_rmse = val_rmse
        best_epoch = epoch
        bad_epochs = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": opt.state_dict(),
                "best_rmse": best_rmse,
                "meta": {
                    "site": SITE,
                    "patch": PATCH,
                    "L": L,
                    "H": H,
                    "seed": SEED,
                    "arch": "SmallResNetEncoder + LSTM + MLP head",
                },
            },
            BEST_PATH,
        )
    else:
        bad_epochs += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train_mse={epoch_out['train_mse']:.5f} | "
        f"val_rmse={val_rmse:.5f} | val_mae={val_mae:.5f} | "
        f"time={epoch_out['epoch_seconds']:.1f}s | "
        f"best_rmse={best_rmse:.5f} (epoch {best_epoch}) | "
        f"bad={bad_epochs}/{PATIENCE}"
    )

    if bad_epochs >= PATIENCE:
        print(f"Early stopping at epoch {epoch}. Best epoch {best_epoch} RMSE={best_rmse:.5f}")
        break

total_train_seconds = float(time.time() - t_train0)
print("Training finished. Total seconds:", round(total_train_seconds, 1))
print("Best checkpoint:", BEST_PATH)

In [ ]:
ckpt = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE)
model.eval()

print("Loaded best model from epoch:", ckpt["epoch"], "| best_rmse:", ckpt["best_rmse"])

final_val  = eval_model(model, val_loader)
final_test = eval_model(model, test_loader)

print("Best-ckpt Model val :", final_val)
print("Best-ckpt Model test:", final_test)

## Summary

In [ ]:
baseline_train = eval_persistence(train_man, ground)
baseline_val   = eval_persistence(val_man, ground)
baseline_test  = eval_persistence(test_man, ground)

summary = {
    "site": SITE,
    "device": DEVICE,
    "seed": SEED,
    "debug": {
        "enabled": bool(DEBUG),
        "train_n": int(len(train_man)),
        "val_n": int(len(val_man)),
        "test_n": int(len(test_man)),
    },
    "data_paths": {
        "site_dir": str(SITE_DIR),
        "mcmipf_root": str(MCMIPF_ROOT),
        "ground_path": str(ground_path),
    },
    "temporal": {
        "freq_min": int(FREQ_MIN),
        "history_steps_L": int(L),
        "horizon_steps_H": int(H),
        "history_hours": float(L * FREQ_MIN / 60.0),
        "horizon_hours": float(H * FREQ_MIN / 60.0),
    },
    "spatial": {
        "grid_size": int(GRID_SIZE),
        "patch": int(PATCH),
        "site_center_rc": {"row": int(SITE_CENTER[0]), "col": int(SITE_CENTER[1])},
        "channels": 16,
    },
    "target_norm": {
        "y_mean_train": float(Y_MEAN),
        "y_std_train": float(Y_STD),
        "y_percentiles_train": np.percentile(y_train, [0, 50, 90, 95, 99]).tolist(),
    },
    "baselines": {
        "persistence_train": baseline_train,
        "persistence_val": baseline_val,
        "persistence_test": baseline_test,
    },
    "model": {
        "arch": "SmallResNetEncoder + LSTM + MLP head",
        "in_ch": 16,
        "patch": int(PATCH),
        "emb_dim": 128,
        "hidden_t": 128,
        "optimizer": "Adam",
        "lr": 1e-3,
        "batch_size": int(BATCH_SIZE),
        "num_workers": int(NUM_WORKERS),
        "epochs": int(EPOCHS),
        "train_seconds_total": float(total_train_seconds),
        "final_val": final_val,
        "final_test": final_test,
        "best_ckpt_path": str(BEST_PATH),
        "best_epoch": int(best_epoch),
        "best_rmse": float(best_rmse),
        "n_params": int(n_params),
    },
    "training_log": train_log,
}

print("=== Baseline vs Model (val/test) ===")
print("Persistence val :", baseline_val)
print("Model val       :", final_val)
print("Persistence test:", baseline_test)
print("Model test      :", final_test)

print("\n=== Full run summary JSON ===")
print(json.dumps(summary, indent=2))

In [ ]:
RUNS_DIR = PROJECT_ROOT / "runs" / "resnet_lstm"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

run_name = f"{SITE}_H{H}_L{L}_P{PATCH}_seed{SEED}_{pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')}"
out_path = RUNS_DIR / f"{run_name}.json"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved run summary:", out_path)